Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 1
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC query = """
# MAGIC
# MAGIC WITH last_audit_trail AS (
# MAGIC     SELECT
# MAGIC         claim_id,
# MAGIC         MAX(from_iso8601_timestamp(audit_time)) AS last_date_trail
# MAGIC     FROM gi_claims.claim_audit_trail_v2
# MAGIC     WHERE claim_id IN (SELECT id FROM gi_claims.health_claims WHERE type = 'LOAN_ATTACH')
# MAGIC     GROUP BY claim_id
# MAGIC ),
# MAGIC
# MAGIC ldr_table AS (
# MAGIC     SELECT
# MAGIC         claim_id,
# MAGIC         MAX(from_iso8601_timestamp(audit_time)) AS ldr_time
# MAGIC     FROM gi_claims.claim_audit_trail_v2
# MAGIC     WHERE from_status = 'MISSING_INFO'
# MAGIC       AND to_status = 'PENDING_ADJUDICATION'
# MAGIC     GROUP BY claim_id
# MAGIC )
# MAGIC
# MAGIC SELECT
# MAGIC     hc.id,
# MAGIC     hc.policy_id,
# MAGIC     hc.member_id,
# MAGIC     hc.sub_type,
# MAGIC     hc.status,
# MAGIC     date_format(from_iso8601_timestamp(hc.date_of_admission), '%d-%b-%Y') AS date_of_admission,
# MAGIC     date_format(from_iso8601_timestamp(hc.date_of_discharge), '%d-%b-%Y') AS date_of_discharge,
# MAGIC     hc.approved_amount,
# MAGIC     hc.claimed_amount,
# MAGIC     JSON_EXTRACT_SCALAR(hc.patient_info, '$.name') AS patient_name,
# MAGIC     JSON_EXTRACT_SCALAR(hc.patient_info, '$.age') AS age,
# MAGIC     date_format(
# MAGIC         date(
# MAGIC             concat(
# MAGIC                 JSON_EXTRACT_SCALAR(hc.patient_info, '$.dob[0]'), '-',
# MAGIC                 lpad(JSON_EXTRACT_SCALAR(hc.patient_info, '$.dob[1]'), 2, '0'), '-',
# MAGIC                 lpad(JSON_EXTRACT_SCALAR(hc.patient_info, '$.dob[2]'), 2, '0')
# MAGIC             )
# MAGIC         ),
# MAGIC         '%d-%b-%Y'
# MAGIC     ) AS dob,
# MAGIC     JSON_EXTRACT_SCALAR(hc.patient_info, '$.gender') AS gender,
# MAGIC     JSON_EXTRACT_SCALAR(hc.patient_info, '$.mobileNumber') AS phone_number,
# MAGIC     JSON_EXTRACT_SCALAR(hc.patient_info, '$.relationship') AS relationship,
# MAGIC     JSON_EXTRACT_SCALAR(hc.hospital_info, '$.city') AS hospital_city,
# MAGIC     JSON_EXTRACT_SCALAR(hc.hospital_info, '$.state') AS hospital_state,
# MAGIC     JSON_EXTRACT_SCALAR(hc.hospital_info, '$.pincode') AS hospital_pincode,
# MAGIC     JSON_EXTRACT_SCALAR(hc.hospital_info, '$.hospitalId') AS hospital_id,
# MAGIC     JSON_EXTRACT_SCALAR(hc.hospital_info, '$.hospitalName') AS hospital_name,
# MAGIC     JSON_EXTRACT_SCALAR(hc.hospital_info, '$.hospitalAddress') AS hospital_address,
# MAGIC     JSON_EXTRACT_SCALAR(hc.financier_details, '$.loanAccountNumber') AS lan_number,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.icdCode') AS icd_code,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.icdGroup') AS icd_group,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.treatment') AS treatment_type,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.icdCategory') AS icd_category,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.ailmentGroup') AS ailment_group,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.lengthOfStay') AS length_of_stay,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.admissionType') AS admission_type,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.finalDiagnosis') AS final_diagnosis,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.perDayRoomRent') AS per_day_roomrent,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.treatingDoctor') AS treating_doctor_name,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.inPatientNumber') AS inPatient_number,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.ailmentDescription') AS ailment_description,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.hospitalisationType') AS hospitalisation_type,
# MAGIC     JSON_EXTRACT_SCALAR(hc.ailment_details, '$.hospitalisationReason') AS hospitalisation_reason,
# MAGIC     date_format(
# MAGIC         date(
# MAGIC             concat(
# MAGIC                 JSON_EXTRACT_SCALAR(hc.ailment_details, '$.dateOfDiagnosis[0]'), '-',
# MAGIC                 lpad(JSON_EXTRACT_SCALAR(hc.ailment_details, '$.dateOfDiagnosis[1]'), 2, '0'), '-',
# MAGIC                 lpad(JSON_EXTRACT_SCALAR(hc.ailment_details, '$.dateOfDiagnosis[2]'), 2, '0')
# MAGIC             )
# MAGIC         ),
# MAGIC         '%d-%b-%Y'
# MAGIC     ) AS date_of_diagnosis,
# MAGIC     JSON_EXTRACT_SCALAR(hc.rejection_details, '$.irdaiDetails.irdaiLevelOne') AS irdai_level_one,
# MAGIC     JSON_EXTRACT_SCALAR(hc.rejection_details, '$.irdaiDetails.irdaiLevelTwo') AS irdai_level_two,
# MAGIC     JSON_EXTRACT_SCALAR(hc.rejection_details, '$.rejectionReasons[0].adjudicatorRemarks') AS adjudicator_remarks,
# MAGIC     JSON_EXTRACT_SCALAR(hc.rejection_details, '$.rejectionReasons[0].policyTermReference') AS policy_term_reference,
# MAGIC     JSON_EXTRACT_SCALAR(hc.rejection_details, '$.letterBodyTextCustomer') AS letterbody_text_customer,
# MAGIC     p.application_type AS Product_name,
# MAGIC     p.product_plan_id,
# MAGIC     p.policy_process_type,
# MAGIC     p.sum_insured,
# MAGIC     p.pin_code AS policy_holder_pincode,
# MAGIC     p.id AS Base_policy_id,
# MAGIC     date_format(from_iso8601_timestamp(p.issued_date), '%d-%b-%Y') AS policy_start_date,
# MAGIC     date_format(from_iso8601_timestamp(p.end_date), '%d-%b-%Y') AS policy_end_date,
# MAGIC     p.status AS policy_status,
# MAGIC     pc.state_name AS policy_holder_state,
# MAGIC     pc.city AS policy_holder_city,
# MAGIC     pc.district AS policy_holder_district,
# MAGIC     date_format(from_iso8601_timestamp(cat_created.audit_time), '%d-%b-%Y') AS created_date,
# MAGIC     date_format(last_audit_trail.last_date_trail, '%d-%b-%Y') AS last_date_trail,
# MAGIC     COALESCE(mi.missing_info_raised, 'no') AS missing_info_raised,
# MAGIC     COALESCE(inv.investigation_raised, 'no') AS investigation_raised,
# MAGIC     CASE
# MAGIC         WHEN mi.missing_info_raised = 'yes' THEN date_format(ldr_table.ldr_time, '%d-%b-%Y')
# MAGIC         ELSE date_format(from_iso8601_timestamp(cat_created.audit_time), '%d-%b-%Y')
# MAGIC     END AS LDR
# MAGIC FROM gi_claims.health_claims hc
# MAGIC LEFT JOIN gi_policy_service.policy p
# MAGIC     ON hc.policy_id = p.id
# MAGIC LEFT JOIN gi_product_service.pincodes pc
# MAGIC     ON p.pin_code = pc.pincode
# MAGIC LEFT JOIN gi_claims.claim_audit_trail_v2 cat_created
# MAGIC     ON hc.id = cat_created.claim_id
# MAGIC     AND cat_created.from_status = 'CLAIM_CREATED'
# MAGIC     AND cat_created.to_status = 'PENDING_DATA_ENTRY_ME'
# MAGIC LEFT JOIN last_audit_trail
# MAGIC     ON hc.id = last_audit_trail.claim_id
# MAGIC LEFT JOIN (
# MAGIC     SELECT DISTINCT claim_id, 'yes' AS missing_info_raised
# MAGIC     FROM gi_claims.claim_audit_trail_v2
# MAGIC     WHERE from_status = 'PENDING_ADJUDICATION'
# MAGIC       AND to_status = 'MISSING_INFO'
# MAGIC ) mi
# MAGIC     ON hc.id = mi.claim_id
# MAGIC LEFT JOIN (
# MAGIC     SELECT DISTINCT claim_id, 'yes' AS investigation_raised
# MAGIC     FROM gi_claims.claim_audit_trail_v2
# MAGIC     WHERE from_status = 'PENDING_ADJUDICATION'
# MAGIC       AND to_status = 'INVESTIGATION'
# MAGIC ) inv
# MAGIC     ON hc.id = inv.claim_id
# MAGIC LEFT JOIN ldr_table
# MAGIC     ON hc.id = ldr_table.claim_id
# MAGIC WHERE hc.type = 'LOAN_ATTACH'
# MAGIC   AND hc.status NOT IN (
# MAGIC       'MISSING_INFO',
# MAGIC       'INVESTIGATION',
# MAGIC       'PENDING_ADJUDICATION',
# MAGIC       'PENDING_INTERNAL_PROCESSING'
# MAGIC   )
# MAGIC
# MAGIC """
# MAGIC
# MAGIC df  = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1Rhd5k9UExwljVYmCjuUr3SSjIXlzLTMVLflX_WMaQT0/edit?gid=1844685988#gid=1844685988"
worksheet_name = "EMI Shield"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)